# Splice Site Prediction Using Foundation Model Embeddings

**Pipeline Overview:**
1. Extract embeddings from foundation models (offline, one-time)
2. Train lightweight 3-class classifiers on embeddings
3. Evaluate using 5-fold cross-validation
4. Compare results across models and window sizes

**Expected duration:** ~3-4 hours total

**Models:** HyenaDNA, DNABert, NucleotideTransformer

**Window sizes:** 300, 600, 1000, 2000, 5000, 10000 bp

**Nucleotide Transformer limits:**
- NT v1 500M runs up to window 6000, so it supports 5000 and skips only window 10000 in this notebook
- NT v2 runs up to window 12000, so it supports all configured windows here

## Setup: Import Libraries and Configuration

In [1]:
import os
import sys
import json
import importlib
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Force-disable flash/triton attention kernels for compatibility
os.environ['FLASH_ATTENTION_DISABLE'] = '1'
os.environ['USE_FLASH_ATTENTION'] = '0'
os.environ['TRITON_DISABLE_LINE_INFO'] = '1'

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

import config
import models
import splicing_embed_extract
import splicing_train
importlib.reload(config)
importlib.reload(models)
importlib.reload(splicing_embed_extract)
importlib.reload(splicing_train)

from config import *
from splicing_embed_extract import EmbeddingExtractor
from splicing_classifier import SpliceSiteClassifier
from splicing_dataset import EmbeddingDataset, create_embedding_dataloader
from splicing_metrics import MetricsComputer
from splicing_train import SpliceClassifierTrainer

print(f"✓ Imports successful (reloaded latest src modules)")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Project directory: {PROJECT_DIR}")
print(f"Window sizes: {WINDOW_SIZES}")

✓ Imports successful (reloaded latest src modules)
Device: NVIDIA GeForce RTX 5080
Project directory: d:\Splice_FMs_seq_lengths
Window sizes: [300, 600, 1000, 2000, 5000, 10000]


## Configuration Check

In [2]:
# Verify data files exist
print("Data files available:")
for ws in WINDOW_SIZES:
    gencode_dir = RAW_DATA_DIR / f'gencode{ws}.csv'
    gtex_dir = GTEX_DATA_DIR / f'gtex{ws}.csv'
    print(f"  Window {ws}:")
    print(f"    GENCODE: {gencode_dir.exists()}")
    print(f"    GTEx: {gtex_dir.exists()}")

print(f"\nOutput directories:")
print(f"  Embeddings: {EMBEDDINGS_DIR}")
print(f"  Results: {SPLICING_RESULTS_DIR}")
print(f"  TensorBoard: {SPLICING_TENSORBOARD_DIR}")

print(f"\nEmbedding extraction config:")
print(f"  Method: {EMBEDDING_CONFIG['method']}")
print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']} (GPU memory safety)")
print(f"  Adaptive batch sizes by window:")
for ws, bs in EMBEDDING_CONFIG.get('batch_size_by_window', {}).items():
    print(f"    Window {ws:5d} bp: batch_size = {bs}")

print(f"\nModel/window support:")
for model_name, model_cfg in MODELS_CONFIG.items():
    for model_id in model_cfg['model_ids']:
        supported = [ws for ws in WINDOW_SIZES if is_model_window_supported(model_name, model_id, ws)]
        skipped = [ws for ws in WINDOW_SIZES if not is_model_window_supported(model_name, model_id, ws)]
        print(f"  {model_id.split('/')[-1]}")
        print(f"    Supported: {supported}")
        if skipped:
            print(f"    Skipped:   {skipped}")

Data files available:
  Window 300:
    GENCODE: True
    GTEx: True
  Window 600:
    GENCODE: True
    GTEx: True
  Window 1000:
    GENCODE: True
    GTEx: True
  Window 2000:
    GENCODE: True
    GTEx: True
  Window 5000:
    GENCODE: True
    GTEx: True
  Window 10000:
    GENCODE: True
    GTEx: True

Output directories:
  Embeddings: d:\Splice_FMs_seq_lengths\data\embeddings
  Results: d:\Splice_FMs_seq_lengths\results\classifiers
  TensorBoard: d:\Splice_FMs_seq_lengths\logs\splicing_classifiers

Embedding extraction config:
  Method: center
  Max length: 10000
  Use FP16: True (GPU memory safety)
  Adaptive batch sizes by window:
    Window   300 bp: batch_size = 256
    Window   600 bp: batch_size = 128
    Window  1000 bp: batch_size = 64
    Window  2000 bp: batch_size = 32
    Window  5000 bp: batch_size = 8
    Window 10000 bp: batch_size = 4

Model/window support:
  hyenadna-small-32k-seqlen-hf
    Supported: [300, 600, 1000, 2000, 5000, 10000]
  hyenadna-medium-160k-se

In [3]:
# QUICK RUN: DNABERT2 CLS only | windows 300 & 600 | embed + train + best-fold test inference
QUICK_WINDOWS = [300]
QUICK_EPOCHS = 100
QUICK_PATIENCE = 5
QUICK_BATCH_SIZE = 256
QUICK_OUTPUT_SUFFIX = "_CLS"


dnabert_only_config = {"DNABert": MODELS_CONFIG["DNABert"]}

print("=" * 80)
print("QUICK DNABERT2 CLS PIPELINE")
print("=" * 80)
print(f"Windows: {QUICK_WINDOWS}")
print(f"Model IDs: {dnabert_only_config['DNABert']['model_ids']}")
print(f"Output suffix: {QUICK_OUTPUT_SUFFIX}")

# 1) Extract embeddings (DNABERT uses CLS pooling in src/splicing_embed_extract.py)
extractor = EmbeddingExtractor(device='cuda' if torch.cuda.is_available() else 'cpu')
quick_extract_stats = extractor.extract_all(
    window_sizes=QUICK_WINDOWS,
    models_config=dnabert_only_config,
    output_suffix=QUICK_OUTPUT_SUFFIX,
)
print("\nExtraction stats:", quick_extract_stats)

# 2) Collect available quick combinations
quick_combinations = []
for ws in QUICK_WINDOWS:
    for model_id in dnabert_only_config['DNABert']['model_ids']:
        combo_name = f"DNABert_{model_id}{QUICK_OUTPUT_SUFFIX}"
        embed_dir = EMBEDDINGS_DIR / str(ws) / combo_name
        trainval_file = embed_dir / "trainval_embeddings.pt"
        test_file = embed_dir / "test_embeddings.pt"
        gtex_file = embed_dir / "gtex_test_embeddings.pt"
        if trainval_file.exists() and test_file.exists() and gtex_file.exists():
            quick_combinations.append({
                'window_size': ws,
                'model_name': 'DNABert',
                'model_id': model_id,
                'combo_name': combo_name,
                'embed_dir': embed_dir,
            })

print(f"\nQuick combinations ready: {len(quick_combinations)}")
for c in quick_combinations:
    print(f"  - w{c['window_size']}: {c['combo_name']}")
    print(f"    dir: {c['embed_dir']}")

# 3) Train quick CV
quick_train_results = {}
for c in quick_combinations:
    exp_name = f"{c['combo_name']}_w{c['window_size']}_quick"
    trainval_data = torch.load(c['embed_dir'] / 'trainval_embeddings.pt', map_location='cpu')
    x = trainval_data['embeddings']
    y = trainval_data['labels']
    if isinstance(x, torch.Tensor):
        x = x.numpy()
    if isinstance(y, torch.Tensor):
        y = y.numpy()

    trainer = SpliceClassifierTrainer(
        embedding_dim=x.shape[1],
        device='cuda' if torch.cuda.is_available() else 'cpu',
        results_dir=str(SPLICING_RESULTS_DIR),
    )
    quick_train_results[exp_name] = trainer.train_with_cv(
        x, y,
        experiment_name=exp_name,
        num_folds=5,
        num_epochs=QUICK_EPOCHS,
        batch_size=QUICK_BATCH_SIZE,
        learning_rate=1e-4,
        weight_decay=1e-3,
        early_stopping_patience=QUICK_PATIENCE,
        seed=42,
        deterministic=False,
    )
    print(f"✓ Trained {exp_name}")

# 4) Best-fold inference on GENCODE test + GTEx test
quick_infer_rows = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _resolve_best_checkpoint(results_dir, cv_results):
    best_fold = cv_results.get('best_fold', {})
    checkpoint_path = best_fold.get('checkpoint_path', None)
    checkpoint_name = best_fold.get('checkpoint', None)
    fold_number = best_fold.get('fold_number', None)

    if checkpoint_name is None:
        per_fold = cv_results.get('per_fold_results', {})
        if not per_fold:
            return None, None
        best_key = max(per_fold, key=lambda k: per_fold[k].get('best_mcc', -1e9))
        checkpoint_name = per_fold[best_key].get('best_checkpoint', None)
        if checkpoint_name is None:
            try:
                fold_idx = int(best_key.replace('fold_', ''))
                checkpoint_name = f"best_fold{fold_idx + 1}.pt"
            except Exception:
                checkpoint_name = None
        try:
            fold_number = int(best_key.replace('fold_', '')) + 1
        except Exception:
            fold_number = None

    if checkpoint_path is not None:
        ckpt = Path(checkpoint_path)
    elif checkpoint_name is not None:
        ckpt = results_dir / 'checkpoints' / checkpoint_name
    else:
        ckpt = None

    if ckpt is None or not ckpt.exists():
        return None, fold_number
    return ckpt, fold_number

def _infer_from_embedding_file(checkpoint_payload, emb_file):
    data = torch.load(emb_file, map_location='cpu')
    emb = data['embeddings']
    labels = data['labels']
    if not isinstance(emb, torch.Tensor):
        emb = torch.tensor(emb, dtype=torch.float32)
    else:
        emb = emb.float()
    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels, dtype=torch.long)
    else:
        labels = labels.long()

    model_state = checkpoint_payload.get('model_state_dict', checkpoint_payload)
    emb_dim = int(checkpoint_payload.get('embedding_dim', emb.shape[1]))
    num_classes = int(checkpoint_payload.get('num_classes', 3))

    clf = SpliceSiteClassifier(
        embedding_dim=emb_dim,
        num_classes=num_classes,
        hidden_dims=[512, 256],
        dropout_rate=0.3,
    ).to(device)
    clf.load_state_dict(model_state)
    clf.eval()

    preds_all, probs_all = [], []
    with torch.no_grad():
        for i in range(0, len(emb), 1024):
            b = emb[i:i+1024].to(device)
            logits = clf(b)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            preds_all.append(preds.cpu())
            probs_all.append(probs.cpu())

    y_true = labels.cpu().numpy()
    y_pred = torch.cat(preds_all, dim=0).numpy()
    y_prob = torch.cat(probs_all, dim=0).numpy()
    metrics, _ = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)
    return metrics

for c in quick_combinations:
    exp_name = f"{c['combo_name']}_w{c['window_size']}_quick"
    results_dir = SPLICING_RESULTS_DIR / exp_name
    results_file = results_dir / 'results.json'
    if not results_file.exists():
        continue
    with open(results_file, 'r') as f:
        cv_results = json.load(f)
    ckpt, fold = _resolve_best_checkpoint(results_dir, cv_results)
    if ckpt is None:
        print(f"WARNING Missing checkpoint for {exp_name}")
        continue

    payload = torch.load(ckpt, map_location='cpu')
    gencode_metrics = _infer_from_embedding_file(payload, c['embed_dir'] / 'test_embeddings.pt')
    gtex_metrics = _infer_from_embedding_file(payload, c['embed_dir'] / 'gtex_test_embeddings.pt')

    quick_infer_rows.append({
        'experiment': exp_name,
        'window': c['window_size'],
        'best_fold': fold,
        'gencode_f1_weighted': gencode_metrics.get('f1_weighted', 0.0),
        'gencode_mcc': gencode_metrics.get('mcc', 0.0),
        'gtex_f1_weighted': gtex_metrics.get('f1_weighted', 0.0),
        'gtex_mcc': gtex_metrics.get('mcc', 0.0),
    })

quick_infer_df = pd.DataFrame(quick_infer_rows).sort_values('window').reset_index(drop=True)
print("\n" + "=" * 80)
print("DNABERT2 QUICK CLS INFERENCE (BEST FOLD)")
print("=" * 80)
print(quick_infer_df.to_string(index=False) if not quick_infer_df.empty else "No quick inference rows produced.")

2026-03-11 01:17:51,690 - splicing_embed_extract - INFO - Configured safe attention backend: flash_sdp=False, mem_efficient_sdp=False, math_sdp=True
2026-03-11 01:17:51,691 - splicing_embed_extract - INFO - 
2026-03-11 01:17:51,691 - splicing_embed_extract - INFO - EMBEDDING EXTRACTION - START
2026-03-11 01:17:51,692 - splicing_embed_extract - INFO - ================================================================================
2026-03-11 01:17:51,692 - splicing_embed_extract - INFO - Window sizes: [300]
2026-03-11 01:17:51,692 - splicing_embed_extract - INFO - Models: ['DNABert']
2026-03-11 01:17:51,693 - splicing_embed_extract - INFO - Device: cuda
2026-03-11 01:17:51,693 - splicing_embed_extract - INFO - Output: d:\Splice_FMs_seq_lengths\data\embeddings
2026-03-11 01:17:51,693 - splicing_embed_extract - INFO - 
################################################################################
2026-03-11 01:17:51,694 - splicing_embed_extract - INFO - WINDOW SIZE: 300
2026-03-11 01:17

QUICK DNABERT2 CLS PIPELINE
Windows: [300]
Model IDs: ['zhihan1996/DNABERT-2-117M']
Output suffix: _CLS

Extraction stats: {'total_started': 1, 'total_succeeded': 0, 'total_failed': 0, 'total_skipped': 1, 'total_skipped_existing': 1, 'total_skipped_unsupported': 0, 'start_time': datetime.datetime(2026, 3, 11, 1, 17, 51, 693975), 'errors': []}

Quick combinations ready: 1
  - w300: DNABert_zhihan1996/DNABERT-2-117M_CLS
    dir: d:\Splice_FMs_seq_lengths\data\embeddings\300\DNABert_zhihan1996\DNABERT-2-117M_CLS


2026-03-11 01:17:52,094 - splicing_train - INFO - 
2026-03-11 01:17:52,095 - splicing_train - INFO - TRAINING: DNABert_zhihan1996/DNABERT-2-117M_CLS_w300_quick
2026-03-11 01:17:52,095 - splicing_train - INFO - ================================================================================
2026-03-11 01:17:52,095 - splicing_train - INFO - Embeddings shape: (718417, 768)
2026-03-11 01:17:52,096 - splicing_train - INFO - Labels shape: (718417,)
2026-03-11 01:17:52,097 - splicing_train - INFO - Label distribution: [239327 242009 237081]
2026-03-11 01:17:52,097 - splicing_train - INFO - Num folds: 5, Epochs: 100, Batch size: 256
2026-03-11 01:17:52,244 - splicing_train - INFO - 
################################################################################
2026-03-11 01:17:52,245 - splicing_train - INFO - FOLD 1/5
2026-03-11 01:17:52,246 - splicing_train - INFO - ################################################################################
2026-03-11 01:17:52,468 - splicing_train - IN

✓ Trained DNABert_zhihan1996/DNABERT-2-117M_CLS_w300_quick

DNABERT2 QUICK CLS INFERENCE (BEST FOLD)
                                      experiment  window  best_fold  gencode_f1_weighted  gencode_mcc  gtex_f1_weighted  gtex_mcc
DNABert_zhihan1996/DNABERT-2-117M_CLS_w300_quick     300          3             0.481113     0.226453          0.464814  0.201128


## PHASE 1: Extract Embeddings (Optional - Run Once)

In [3]:
# Skip this cell if embeddings are already extracted
EXTRACT_EMBEDDINGS = True  # Set to True to extract embeddings
EXTRACT_WINDOW_SIZES = [300, 600, 1000, 2000, 10000]

if EXTRACT_EMBEDDINGS:
    print("Extracting embeddings from foundation models...")
    print("Configuration:")
    print(f"  Window sizes: {EXTRACT_WINDOW_SIZES}")
    print(f"  Method: {EMBEDDING_CONFIG['method']}")
    print(f"  Max length: {EMBEDDING_CONFIG['max_length']}")
    print(f"  Use FP16: {EMBEDDING_CONFIG['use_fp16']}")
    print(f"  Device: {EMBEDDING_CONFIG['device']}")
    print(f"\nAdaptive batch sizes:")
    for ws in EXTRACT_WINDOW_SIZES:
        bs = EMBEDDING_CONFIG.get('batch_size_by_window', {}).get(ws, EMBEDDING_CONFIG['batch_size'])
        print(f"  Window {ws:5d} bp: {bs:3d}")
    print("\nUnsupported NT combinations are skipped automatically:")
    print("  - NT v1 500M: skip window 10000 only")
    print("  - NT v2: no skip for the configured windows [300, 600, 1000, 2000, 10000]")
    print(f"\nThis will take ~2-3 hours on RTX 5080\n")
    
    extractor = EmbeddingExtractor(device='cuda')
    stats = extractor.extract_all(window_sizes=EXTRACT_WINDOW_SIZES)
    
    print(f"\n✓ Embedding extraction completed")
    print(f"  Extracted: {stats['total_succeeded']}")
    print(f"  Skipped existing: {stats.get('total_skipped_existing', 0)}")
    print(f"  Skipped unsupported: {stats.get('total_skipped_unsupported', 0)}")
    print(f"  Failed: {stats['total_failed']}")
else:
    print("Skipping embedding extraction.")
    print("To extract embeddings, set EXTRACT_EMBEDDINGS = True and run this cell.")

2026-03-08 22:44:52,527 - splicing_embed_extract - INFO - Configured safe attention backend: flash_sdp=False, mem_efficient_sdp=False, math_sdp=True
2026-03-08 22:44:52,528 - splicing_embed_extract - INFO - 
2026-03-08 22:44:52,528 - splicing_embed_extract - INFO - EMBEDDING EXTRACTION - START
2026-03-08 22:44:52,528 - splicing_embed_extract - INFO - ================================================================================
2026-03-08 22:44:52,529 - splicing_embed_extract - INFO - Window sizes: [300, 600, 1000, 2000, 10000]
2026-03-08 22:44:52,529 - splicing_embed_extract - INFO - Models: ['HyenaDNA', 'DNABert', 'NucleotideTransformer']
2026-03-08 22:44:52,530 - splicing_embed_extract - INFO - Device: cuda
2026-03-08 22:44:52,530 - splicing_embed_extract - INFO - Output: d:\Splice_FMs_seq_lengths\data\embeddings
2026-03-08 22:44:52,530 - splicing_embed_extract - INFO - 
################################################################################
2026-03-08 22:44:52,531 - spli

Extracting embeddings from foundation models...
Configuration:
  Window sizes: [300, 600, 1000, 2000, 10000]
  Method: center
  Max length: 10000
  Use FP16: True
  Device: cuda

Adaptive batch sizes:
  Window   300 bp: 256
  Window   600 bp: 128
  Window  1000 bp:  64
  Window  2000 bp:  32
  Window 10000 bp:   4

Unsupported NT combinations are skipped automatically:
  - NT v1 500M: skip window 10000 only
  - NT v2: no skip for the configured windows [300, 600, 1000, 2000, 10000]

This will take ~2-3 hours on RTX 5080



2026-03-08 22:44:53,295 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-100m-multi-species/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 22:44:53,296 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-08 22:44:53,328 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/InstaDeepAI/nucleotide-transformer-v2-100m-multi-species/f34324c6fde36a4f635f0f1f06cac5d25acd6798/config.json "HTTP/1.1 200 OK"
2026-03-08 22:44:53,621 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-100m-multi-species/resolve/main/esm_config.py "HTTP/1.1 307 Temporary Redirect"
2026-03-08 22:44:53,668 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/InstaDeepAI/nucleotide-transformer-v2-100m-multi-spe

Loading weights:   0%|          | 0/335 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
Key                                              | Status     |                                                                                             
-------------------------------------------------+------------+---------------------------------------------------------------------------------------------
lm_head.layer_norm.weight                        | UNEXPECTED |                                                                                             
lm_head.dense.weight                             | UNEXPECTED |                                                                                             
lm_head.layer_norm.bias                          | UNEXPECTED |                                                                                             
lm_head.bias                                     | UNEXPECTED |                                                                    

...

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

2026-03-09 07:07:58,030 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/InstaDeepAI/nucleotide-transformer-v2-250m-multi-species "HTTP/1.1 200 OK"
EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
Key                                              | Status     |                                                                                             
-------------------------------------------------+------------+---------------------------------------------------------------------------------------------
lm_head.layer_norm.weight                        | UNEXPECTED |                                                                                             
lm_head.dense.weight                             | UNEXPECTED |                                                                                             
lm_head.layer_norm.bias                          | UNEXPECTED |                                                         

Loading weights:   0%|          | 0/365 [00:00<?, ?it/s]

2026-03-09 07:07:58,635 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/InstaDeepAI/nucleotide-transformer-v2-250m-multi-species/discussions?p=0 "HTTP/1.1 200 OK"
2026-03-09 07:07:58,711 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/InstaDeepAI/nucleotide-transformer-v2-250m-multi-species "HTTP/1.1 200 OK"
EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
Key                                              | Status     |                                                                                             
-------------------------------------------------+------------+---------------------------------------------------------------------------------------------
lm_head.layer_norm.weight                        | UNEXPECTED |                                                                                             
lm_head.dense.weight                             | UNEXPECTED |                              


✓ Embedding extraction completed
  Extracted: 2
  Skipped existing: 27
  Skipped unsupported: 1
  Failed: 0


## PHASE 2: List Available Embeddings

In [4]:
# Check which embeddings are available for CV training
available_combinations = []
unsupported_combinations = []
missing_supported_combinations = []

for window_size in WINDOW_SIZES:
    for model_name in MODELS_CONFIG.keys():
        for model_id in MODELS_CONFIG[model_name]['model_ids']:
            combo_name = f"{model_name}_{model_id}"
            
            if not is_model_window_supported(model_name, model_id, window_size):
                unsupported_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                    'reason': get_model_window_skip_reason(model_name, model_id, window_size),
                })
                continue

            embed_dir = EMBEDDINGS_DIR / str(window_size) / combo_name
            trainval_file = embed_dir / "trainval_embeddings.pt"
            test_file = embed_dir / "test_embeddings.pt"
            gtex_file = embed_dir / "gtex_test_embeddings.pt"
            
            if trainval_file.exists() and test_file.exists() and gtex_file.exists():
                available_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                    'combo_name': combo_name,
                    'embed_dir': embed_dir
                })
            else:
                missing_supported_combinations.append({
                    'window_size': window_size,
                    'model_name': model_name,
                    'model_id': model_id,
                })

print(f"Available embeddings for CV training: {len(available_combinations)}")
for combo in available_combinations:
    print(f"  ✓ Window {combo['window_size']:5d} | {combo['combo_name']:30s}")

print(f"\nUnsupported combinations skipped by design: {len(unsupported_combinations)}")
for combo in unsupported_combinations:
    print(f"  ↷ Window {combo['window_size']:5d} | {combo['model_id']} | {combo['reason']}")

if missing_supported_combinations:
    print(f"\nSupported combinations still missing embeddings: {len(missing_supported_combinations)}")
    for combo in missing_supported_combinations:
        print(f"  - Window {combo['window_size']:5d} | {combo['model_id']}")

if len(available_combinations) == 0:
    print("\n⚠ No supported trainval/test embeddings found. Please run embedding extraction first.")

Available embeddings for CV training: 29
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   300 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   300 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window   300 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  ✓ Window   600 | HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  ✓ Window   600 | DNABert_zhihan1996/DNABERT-2-117M
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
  ✓ Window   600 | NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species
  ✓

## PHASE 3: Train Classifiers with 5-Fold CV

In [5]:
# Train classifiers for each supported combination
all_results = {}
training_start_time = datetime.now()

# Skip training if result file already exists
SKIP_IF_RESULTS_EXIST = True

for idx, combo in enumerate(available_combinations):
    if not is_model_window_supported(combo['model_name'], combo['model_id'], combo['window_size']):
        print(
            f"\n↷ Skipped unsupported combination: {combo['combo_name']} "
            f"(Window {combo['window_size']})"
        )
        continue

    print(f"\n{'='*80}")
    print(f"[{idx+1}/{len(available_combinations)}] Training: {combo['combo_name']} (Window {combo['window_size']})")
    print(f"{'='*80}")
    
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / "results.json"

    # Skip if already trained, but still load into all_results for downstream summary
    if SKIP_IF_RESULTS_EXIST and results_file.exists():
        try:
            with open(results_file, 'r') as f:
                existing_results = json.load(f)
            all_results[experiment_name] = existing_results
            print(f"↷ Skipped (already trained): {experiment_name}")
            print(f"  Loaded existing results: {results_file}")
            continue
        except Exception as e:
            print(f"⚠ Found existing results but failed to load ({e}), retraining...")
    
    # Load train+val embeddings prepared for CV
    trainval_file = combo['embed_dir'] / "trainval_embeddings.pt"
    trainval_data = torch.load(trainval_file)
    
    all_embeddings = trainval_data['embeddings']
    all_labels = trainval_data['labels']
    
    embedding_dim = all_embeddings.shape[1]
    
    # Train with CV
    trainer = SpliceClassifierTrainer(
        embedding_dim=embedding_dim,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        results_dir=str(SPLICING_RESULTS_DIR)
    )
    
    results = trainer.train_with_cv(
        all_embeddings.numpy(),
        all_labels.numpy(),
        experiment_name=experiment_name,
        num_folds=5,
        num_epochs=50,
        batch_size=256,
        learning_rate=0.0001,
        weight_decay=0.001,
        early_stopping_patience=5,
        seed=42,
        deterministic=False
    )
    
    all_results[experiment_name] = results
    
    print(f"✓ {experiment_name} training completed")

elapsed_hours = (datetime.now() - training_start_time).total_seconds() / 3600
print(f"\n{'='*80}")
print(f"Total training time: {elapsed_hours:.1f} hours")
print(f"Total experiments available in all_results: {len(all_results)}")
print(f"{'='*80}")


[1/29] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 300)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-small-32k-seqlen-hf_w300\results.json

[2/29] Training: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf (Window 300)
↷ Skipped (already trained): HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\HyenaDNA_LongSafari\hyenadna-medium-160k-seqlen-hf_w300\results.json

[3/29] Training: DNABert_zhihan1996/DNABERT-2-117M (Window 300)
↷ Skipped (already trained): DNABert_zhihan1996/DNABERT-2-117M_w300
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\DNABert_zhihan1996\DNABERT-2-117M_w300\results.json

[4/29] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref (Window 300)
↷ Skipped (already trained)

2026-03-09 17:36:19,708 - splicing_train - INFO - 
2026-03-09 17:36:19,708 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w2000
2026-03-09 17:36:19,708 - splicing_train - INFO - ================================================================================
2026-03-09 17:36:19,708 - splicing_train - INFO - Embeddings shape: (718253, 1280)
2026-03-09 17:36:19,708 - splicing_train - INFO - Labels shape: (718253,)
2026-03-09 17:36:19,708 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-09 17:36:19,708 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-09 17:36:19,781 - splicing_train - INFO - 
################################################################################
2026-03-09 17:36:19,781 - splicing_train - INFO - FOLD 1/5
2026-03-09 17:36:19,781 - splicing_train - INFO - ################################################################################
2026-03-09 17:36

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w2000 training completed

[23/29] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species (Window 2000)
↷ Skipped (already trained): NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w2000
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\NucleotideTransformer_InstaDeepAI\nucleotide-transformer-v2-100m-multi-species_w2000\results.json

[24/29] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species (Window 2000)
↷ Skipped (already trained): NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w2000
  Loaded existing results: d:\Splice_FMs_seq_lengths\results\classifiers\NucleotideTransformer_InstaDeepAI\nucleotide-transformer-v2-250m-multi-species_w2000\results.json

[25/29] Training: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf (Window 10000)
↷ Skipped (already train

2026-03-09 17:46:34,548 - splicing_train - INFO - 
2026-03-09 17:46:34,553 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w10000
2026-03-09 17:46:34,554 - splicing_train - INFO - ================================================================================
2026-03-09 17:46:34,555 - splicing_train - INFO - Embeddings shape: (718253, 512)
2026-03-09 17:46:34,555 - splicing_train - INFO - Labels shape: (718253,)
2026-03-09 17:46:34,556 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-09 17:46:34,557 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-09 17:46:34,619 - splicing_train - INFO - 
################################################################################
2026-03-09 17:46:34,620 - splicing_train - INFO - FOLD 1/5
2026-03-09 17:46:34,620 - splicing_train - INFO - ################################################################################
2026-03-0

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w10000 training completed

[29/29] Training: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species (Window 10000)


2026-03-09 18:22:20,415 - splicing_train - INFO - 
2026-03-09 18:22:20,416 - splicing_train - INFO - TRAINING: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w10000
2026-03-09 18:22:20,417 - splicing_train - INFO - ================================================================================
2026-03-09 18:22:20,417 - splicing_train - INFO - Embeddings shape: (718253, 768)
2026-03-09 18:22:20,418 - splicing_train - INFO - Labels shape: (718253,)
2026-03-09 18:22:20,421 - splicing_train - INFO - Label distribution: [239323 241929 237001]
2026-03-09 18:22:20,421 - splicing_train - INFO - Num folds: 5, Epochs: 50, Batch size: 256
2026-03-09 18:22:20,489 - splicing_train - INFO - 
################################################################################
2026-03-09 18:22:20,490 - splicing_train - INFO - FOLD 1/5
2026-03-09 18:22:20,491 - splicing_train - INFO - ################################################################################
2026-03-0

✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w10000 training completed

Total training time: 1.4 hours
Total experiments available in all_results: 29


## PHASE 4: Results Comparison and Analysis

In [6]:
# Summarize results + export aggregate files
summary_data = []

summary_columns = [
    'Experiment',
    'Window',
    'Accuracy',
    'Balanced Acc',
    'F1 (weighted)',
    'F1 (macro)',
    'MCC',
    'ROC-AUC',
]

for exp_name, results in all_results.items():
    avg_metrics = results.get('averaged_metrics', {})
    if not isinstance(avg_metrics, dict):
        avg_metrics = {}
    
    # Parse window size from experiment name suffix: ..._w300
    window_size = None
    if '_w' in exp_name:
        try:
            window_size = int(exp_name.rsplit('_w', 1)[-1])
        except ValueError:
            window_size = None

    summary_data.append({
        'Experiment': exp_name,
        'Window': window_size,
        'Accuracy': avg_metrics.get('accuracy_mean', 0),
        'Balanced Acc': avg_metrics.get('balanced_accuracy_mean', 0),
        'F1 (weighted)': avg_metrics.get('f1_weighted_mean', 0),
        'F1 (macro)': avg_metrics.get('f1_macro_mean', 0),
        'MCC': avg_metrics.get('mcc_mean', 0),
        'ROC-AUC': avg_metrics.get('roc_auc_weighted_mean', 0),
    })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    for col in summary_columns:
        if col not in summary_df.columns:
            summary_df[col] = np.nan
    summary_df = summary_df[summary_columns]
    summary_df = summary_df.sort_values('F1 (weighted)', ascending=False, na_position='last')
else:
    summary_df = pd.DataFrame(columns=summary_columns)

print("\n" + "="*100)
print("RESULTS SUMMARY (Sorted by F1-Weighted)")
print("="*100)
if summary_df.empty:
    print("No results available in all_results. Run PHASE 3 first or check loaded results files.")
else:
    print(summary_df.to_string(index=False))
print("="*100)

# Export aggregate summaries
summary_dir = SPLICING_RESULTS_DIR / 'summaries'
summary_dir.mkdir(parents=True, exist_ok=True)

all_csv = summary_dir / 'all_experiments_summary.csv'
all_json = summary_dir / 'all_experiments_summary.json'
summary_df.to_csv(all_csv, index=False)
summary_df.to_json(all_json, orient='records', indent=2)

print(f"\nSaved aggregate summaries:")
print(f"  - {all_csv}")
print(f"  - {all_json}")

# Export one file per window
windows_exported = []
if not summary_df.empty and 'Window' in summary_df.columns:
    unique_windows = [w for w in summary_df['Window'].dropna().unique()]
    for ws in sorted(unique_windows):
        ws_int = int(ws)
        ws_df = summary_df[summary_df['Window'] == ws].copy()
        ws_df = ws_df.sort_values('F1 (weighted)', ascending=False, na_position='last')
        ws_csv = summary_dir / f'summary_w{ws_int}.csv'
        ws_json = summary_dir / f'summary_w{ws_int}.json'
        ws_df.to_csv(ws_csv, index=False)
        ws_df.to_json(ws_json, orient='records', indent=2)
        windows_exported.append((ws_csv, ws_json))

if windows_exported:
    print("\nSaved per-window summaries:")
    for ws_csv, ws_json in windows_exported:
        print(f"  - {ws_csv}")
        print(f"  - {ws_json}")


RESULTS SUMMARY (Sorted by F1-Weighted)
                                                                           Experiment  Window  Accuracy  Balanced Acc  F1 (weighted)  F1 (macro)      MCC  ROC-AUC
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w2000    2000  0.922012      0.921990       0.921860    0.921810 0.883084 0.986274
         NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w600     600  0.919324      0.919305       0.919226    0.919169 0.879048 0.985398
         NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w300     300  0.909771      0.909743       0.909545    0.909466 0.864782 0.981873
        NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w1000    1000  0.890775      0.890805       0.890460    0.890473 0.836253 0.975664
 NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w2000    2000  0.834656      0.834478       0.833890    0.833593

## PHASE 5: Best Model Evaluation on Test Set

In [7]:
# Inference with best-fold checkpoint for each experiment on GENCODE test and GTEx test embeddings
print("\nRunning inference using best-fold checkpoints...\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INFER_BATCH_SIZE = 1024
inference_results = {}

def _to_serializable(obj):
    if isinstance(obj, dict):
        return {k: _to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_serializable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def _resolve_best_checkpoint(results_dir, cv_results):
    best_fold_info = cv_results.get('best_fold', {})

    checkpoint_path = best_fold_info.get('checkpoint_path', None)
    checkpoint_name = best_fold_info.get('checkpoint', None)
    fold_number = best_fold_info.get('fold_number', None)

    # Fallback for old results format without best_fold
    if checkpoint_name is None:
        fold_entries = cv_results.get('per_fold_results', {})
        if not fold_entries:
            return None, None, None

        best_key = None
        best_mcc = -1e9
        for fold_key, fold_val in fold_entries.items():
            mcc = fold_val.get('best_mcc', -1e9)
            if mcc > best_mcc:
                best_mcc = mcc
                best_key = fold_key

        if best_key is None:
            return None, None, None

        checkpoint_name = fold_entries[best_key].get('best_checkpoint', None)
        if checkpoint_name is None:
            # If old results do not contain checkpoint name, infer from fold index
            try:
                fold_idx = int(best_key.replace('fold_', ''))
                checkpoint_name = f"best_fold{fold_idx + 1}.pt"
            except Exception:
                checkpoint_name = None

        try:
            fold_number = int(best_key.replace('fold_', '')) + 1
        except Exception:
            fold_number = None

    checkpoint_path_obj = None
    if checkpoint_path is not None:
        checkpoint_path_obj = Path(checkpoint_path)
        if not checkpoint_path_obj.exists():
            checkpoint_path_obj = None

    if checkpoint_path_obj is None and checkpoint_name is not None:
        checkpoint_path_obj = results_dir / 'checkpoints' / checkpoint_name

    if checkpoint_path_obj is None or not checkpoint_path_obj.exists():
        return None, checkpoint_name, fold_number

    return checkpoint_path_obj, checkpoint_name, fold_number

def _run_inference_on_embedding_file(checkpoint_payload, embedding_file, batch_size=1024):
    data = torch.load(embedding_file, map_location='cpu')
    embeddings = data['embeddings']
    labels = data['labels']

    if not isinstance(embeddings, torch.Tensor):
        embeddings = torch.tensor(embeddings, dtype=torch.float32)
    else:
        embeddings = embeddings.float()

    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels, dtype=torch.long)
    else:
        labels = labels.long()

    model_state = checkpoint_payload.get('model_state_dict', checkpoint_payload)
    embedding_dim = int(checkpoint_payload.get('embedding_dim', embeddings.shape[1]))
    num_classes = int(checkpoint_payload.get('num_classes', 3))

    model = SpliceSiteClassifier(
        embedding_dim=embedding_dim,
        num_classes=num_classes,
        hidden_dims=[512, 256],
        dropout_rate=0.3,
    ).to(device)
    model.load_state_dict(model_state)
    model.eval()

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for start_idx in range(0, len(embeddings), batch_size):
            end_idx = min(start_idx + batch_size, len(embeddings))
            batch_x = embeddings[start_idx:end_idx].to(device)
            logits = model(batch_x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())

    y_true = labels.cpu().numpy()
    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_prob = torch.cat(all_probs, dim=0).numpy()

    metrics, cm = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)
    return {
        'num_samples': int(len(y_true)),
        'metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in metrics.items()},
        'confusion_matrix': cm.tolist(),
    }

for combo in available_combinations:
    experiment_name = f"{combo['combo_name']}_w{combo['window_size']}"
    results_dir = SPLICING_RESULTS_DIR / experiment_name
    results_file = results_dir / 'results.json'

    if not results_file.exists():
        print(f"⚠ Missing results.json: {results_file}")
        continue

    with open(results_file, 'r') as f:
        cv_results = json.load(f)

    checkpoint_path, checkpoint_name, fold_number = _resolve_best_checkpoint(results_dir, cv_results)
    if checkpoint_path is None:
        print(f"⚠ Missing best-fold checkpoint for {experiment_name}")
        continue

    test_embed_file = combo['embed_dir'] / 'test_embeddings.pt'
    gtex_embed_file = combo['embed_dir'] / 'gtex_test_embeddings.pt'

    if not test_embed_file.exists() or not gtex_embed_file.exists():
        print(f"⚠ Missing test embedding files for {experiment_name}")
        continue

    checkpoint_payload = torch.load(checkpoint_path, map_location='cpu')

    gencode_test_eval = _run_inference_on_embedding_file(
        checkpoint_payload=checkpoint_payload,
        embedding_file=test_embed_file,
        batch_size=INFER_BATCH_SIZE,
    )
    gtex_test_eval = _run_inference_on_embedding_file(
        checkpoint_payload=checkpoint_payload,
        embedding_file=gtex_embed_file,
        batch_size=INFER_BATCH_SIZE,
    )

    inference_results[experiment_name] = {
        'window_size': combo['window_size'],
        'model_name': combo['model_name'],
        'model_id': combo['model_id'],
        'best_fold_number': fold_number,
        'checkpoint': checkpoint_name,
        'checkpoint_path': str(checkpoint_path),
        'gencode_test': gencode_test_eval,
        'gtex_test': gtex_test_eval,
    }

    print(
        f"✓ {experiment_name} | fold={fold_number} | "
        f"GENCODE F1={gencode_test_eval['metrics'].get('f1_weighted', 0):.4f}, "
        f"GTEx F1={gtex_test_eval['metrics'].get('f1_weighted', 0):.4f}"
    )

print(f"\n✓ Inference completed for {len(inference_results)} experiments")

# Save inference summary
inference_summary_dir = SPLICING_RESULTS_DIR / 'summaries'
inference_summary_dir.mkdir(parents=True, exist_ok=True)
inference_results_file = inference_summary_dir / 'best_fold_inference_results.json'
with open(inference_results_file, 'w') as f:
    json.dump(_to_serializable(inference_results), f, indent=2)

print(f"Saved inference results: {inference_results_file}")


Running inference using best-fold checkpoints...

✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w300 | fold=3 | GENCODE F1=0.7732, GTEx F1=0.7877
✓ HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w300 | fold=5 | GENCODE F1=0.8008, GTEx F1=0.8127
✓ DNABert_zhihan1996/DNABERT-2-117M_w300 | fold=5 | GENCODE F1=0.5801, GTEx F1=0.5775
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w300 | fold=3 | GENCODE F1=0.9057, GTEx F1=0.9076
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-100m-multi-species_w300 | fold=2 | GENCODE F1=0.7895, GTEx F1=0.8012
✓ NucleotideTransformer_InstaDeepAI/nucleotide-transformer-v2-250m-multi-species_w300 | fold=2 | GENCODE F1=0.7779, GTEx F1=0.7836
✓ HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf_w600 | fold=5 | GENCODE F1=0.7818, GTEx F1=0.7960
✓ HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w600 | fold=5 | GENCODE F1=0.8183, GTEx F1=0.8317
✓ DNABert_zhihan1996/DNABERT-2-117M_w600 | fold=1 | GENCODE F1=0.5452

## Summary

In [8]:
print("\n" + "="*80)
print("BEST FOLD INFERENCE SUMMARY")
print("="*80)

if 'inference_results' not in globals() or not inference_results:
    print("Run Phase 5 first to generate best-fold inference results.")
else:
    best_gencode = max(
        inference_results.items(),
        key=lambda item: item[1]['gencode_test']['metrics'].get('f1_weighted', float('-inf'))
    )
    best_gtex = max(
        inference_results.items(),
        key=lambda item: item[1]['gtex_test']['metrics'].get('f1_weighted', float('-inf'))
    )

    best_gencode_name, best_gencode_result = best_gencode
    best_gtex_name, best_gtex_result = best_gtex

    print("\nBest on GENCODE test:")
    print(f"  {best_gencode_name}")
    print(f"    Fold: {best_gencode_result['best_fold_number']}")
    print(f"    F1-weighted: {best_gencode_result['gencode_test']['metrics'].get('f1_weighted', 0):.4f}")
    print(f"    Accuracy: {best_gencode_result['gencode_test']['metrics'].get('accuracy', 0):.4f}")
    print(f"    MCC: {best_gencode_result['gencode_test']['metrics'].get('mcc', 0):.4f}")

    print("\nBest on GTEx test:")
    print(f"  {best_gtex_name}")
    print(f"    Fold: {best_gtex_result['best_fold_number']}")
    print(f"    F1-weighted: {best_gtex_result['gtex_test']['metrics'].get('f1_weighted', 0):.4f}")
    print(f"    Accuracy: {best_gtex_result['gtex_test']['metrics'].get('accuracy', 0):.4f}")
    print(f"    MCC: {best_gtex_result['gtex_test']['metrics'].get('mcc', 0):.4f}")

    print("\nSaved inference results:")
    print(f"  {SPLICING_RESULTS_DIR / 'summaries' / 'best_fold_inference_results.json'}")

print(f"\n" + "="*80)


BEST FOLD INFERENCE SUMMARY

Best on GENCODE test:
  NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w2000
    Fold: 1
    F1-weighted: 0.9188
    Accuracy: 0.9188
    MCC: 0.8782

Best on GTEx test:
  NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref_w600
    Fold: 1
    F1-weighted: 0.9147
    Accuracy: 0.9147
    MCC: 0.8720

Saved inference results:
  d:\Splice_FMs_seq_lengths\results\classifiers\summaries\best_fold_inference_results.json



In [3]:
import json
import pandas as pd

def extract_macro_metrics(json_path):
    # Đọc file JSON
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Không tìm thấy file: {json_path}")
        return None

    records = []

    # Lặp qua từng experiment (tổ hợp model + window size)
    for exp_key, exp_info in data.items():
        model_name = exp_info.get("model_name", "Unknown")
        model_id = exp_info.get("model_id", "Unknown") # Lấy thêm version của model
        window_size = exp_info.get("window_size", "Unknown")

        # Hàm phụ để xử lý logic lấy data cho từng tập test
        def get_dataset_metrics(dataset_key, prefix):
            test_data = exp_info.get(dataset_key, {})
            metrics = test_data.get("metrics", {})

            if not metrics:
                return {f"{prefix}_F1": "N/A", f"{prefix}_MCC": "N/A", 
                        f"{prefix}_AUC": "N/A", f"{prefix}_AUPRC": "N/A"}

            # Lấy thẳng các metric hệ Macro
            f1_macro = metrics.get("f1_macro", None)
            mcc = metrics.get("mcc", None)
            auc_macro = metrics.get("roc_auc_macro", None)

            # Tính Macro AUPRC bằng trung bình cộng của Other, Donor, Acceptor
            pr_other = metrics.get("pr_auc_Other", 0)
            pr_donor = metrics.get("pr_auc_Donor", 0)
            pr_acceptor = metrics.get("pr_auc_Acceptor", 0)

            if pr_other and pr_donor and pr_acceptor:
                auprc_macro = (pr_other + pr_donor + pr_acceptor) / 3.0
            else:
                auprc_macro = None

            return {
                f"{prefix}_F1": f"{f1_macro:.4f}" if f1_macro else "N/A",
                f"{prefix}_MCC": f"{mcc:.4f}" if mcc else "N/A",
                f"{prefix}_AUC": f"{auc_macro:.4f}" if auc_macro else "N/A",
                f"{prefix}_AUPRC": f"{auprc_macro:.4f}" if auprc_macro else "N/A"
            }

        # Trích xuất dữ liệu cho Gencode và GTEx
        gencode_metrics = get_dataset_metrics("gencode_test", "Gencode")
        gtex_metrics = get_dataset_metrics("gtex_test", "GTEx")

        # Gom thành 1 dòng (row), thêm cột Version
        row = {
            "Model": model_name,
            "Version": model_id,
            "Window Size": window_size,
            **gencode_metrics,
            **gtex_metrics
        }
        records.append(row)

    # Tạo DataFrame và sắp xếp theo Model -> Version -> Window Size
    df = pd.DataFrame(records)
    df = df.sort_values(by=["Model", "Version", "Window Size"]).reset_index(drop=True)
    return df


json_file = r'C:\Users\dotru\STUDIE\FPTU\AiTA_Lab\Splice_FMs_seq_lengths\results\classifiers\summaries\best_fold_inference_results.json' 

df_results = extract_macro_metrics(json_file)
    
if df_results is not None:
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)
    
    print("\n" + "="*140)
    print("SPLICE SITE PREDICTION RESULTS (MACRO METRICS)")
    print("="*140)
    print(df_results.to_string(index=False))
    print("="*140)


SPLICE SITE PREDICTION RESULTS (MACRO METRICS)
                Model                                                  Version  Window Size Gencode_F1 Gencode_MCC Gencode_AUC Gencode_AUPRC GTEx_F1 GTEx_MCC GTEx_AUC GTEx_AUPRC
              DNABert                                zhihan1996/DNABERT-2-117M          300     0.5804      0.3718      0.7663        0.6340  0.5775   0.3677   0.7634     0.6230
              DNABert                                zhihan1996/DNABERT-2-117M          600     0.5455      0.3230      0.7345        0.5911  0.5447   0.3204   0.7315     0.5837
              DNABert                                zhihan1996/DNABERT-2-117M         1000     0.5171      0.2813      0.7051        0.5560  0.5173   0.2817   0.7055     0.5535
              DNABert                                zhihan1996/DNABERT-2-117M         2000     0.4848      0.2315      0.6722        0.5125  0.4769   0.2202   0.6640     0.5015
              DNABert                                zhihan199

# Mở Tensorboard
### To monitor training progress in real-time and view test results:

```bash
tensorboard --logdir results\classifiers
```

Then open the URL in your browser to view:
- Training/Validation loss curves
- Metric evaluation (accuracy, precision, recall, F1, MCC, auc, balanced acc, specificity)
- Model architecture summary